In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn huggingface_hub --quiet

: 

In [ ]:
import numpy as np
import pandas as pd
import os

np.random.seed(42)
n = 300

size_sqft      = np.random.uniform(500, 4000, n)
bedrooms       = np.random.randint(1, 7, n)
bathrooms      = np.random.randint(1, 5, n)
age_years      = np.random.randint(0, 41, n)
location_score = np.random.randint(1, 11, n)

price_lakhs = (
    15.0
    + (size_sqft      * 0.05)
    + (bedrooms       * 5.0)
    + (bathrooms      * 3.0)
    + (location_score * 4.0)
    - (age_years      * 0.5)
)

noise = np.random.normal(0, 12, n)
price_lakhs = np.clip(price_lakhs + noise, 10.0, None)

df = pd.DataFrame({
    'size_sqft':      np.round(size_sqft, 2),
    'bedrooms':       bedrooms,
    'bathrooms':      bathrooms,
    'age_years':      age_years,
    'location_score': location_score,
    'price_lakhs':    np.round(price_lakhs, 2)
})

os.makedirs('data', exist_ok=True)
df.to_csv('data/house_prices.csv', index=False)
df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df['price_lakhs'], bins=25, color='#4F46E5', alpha=0.8, edgecolor='white')
axes[0].set_title('Price Distribution (Lakhs)')
axes[0].set_xlabel('Price (Rs. Lakhs)')
axes[0].set_ylabel('Count')

axes[1].scatter(df['size_sqft'], df['price_lakhs'], alpha=0.4, color='#4F46E5', s=20)
axes[1].set_title('Size vs Price')
axes[1].set_xlabel('Size (sq ft)')
axes[1].set_ylabel('Price (Rs. Lakhs)')

axes[2].scatter(df['location_score'], df['price_lakhs'], alpha=0.4, color='#059669', s=20)
axes[2].set_title('Location Score vs Price')
axes[2].set_xlabel('Location Score (1-10)')
axes[2].set_ylabel('Price (Rs. Lakhs)')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import json

features = ['size_sqft', 'bedrooms', 'bathrooms', 'age_years', 'location_score']
target   = 'price_lakhs'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

print('Model coefficients:', model.coef_)
print('Model intercept:', model.intercept_)

In [ ]:
true_coefs = {
    'size_sqft': 0.05, 'bedrooms': 5.0, 'bathrooms': 3.0,
    'age_years': -0.5, 'location_score': 4.0
}

x_pos = range(len(features))
fig, ax = plt.subplots(figsize=(10, 5))
bar_w = 0.35
ax.bar([p - bar_w/2 for p in x_pos], [true_coefs[f] for f in features], width=bar_w, label='True', color='#6EE7B7')
ax.bar([p + bar_w/2 for p in x_pos], model.coef_, width=bar_w, label='Learned', color='#4F46E5', alpha=0.85)
ax.set_xticks(list(x_pos))
ax.set_xticklabels(features)
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_title('True vs Learned Coefficients')
ax.legend()
plt.show()

In [ ]:
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f'MAE  : Rs. {mae:.2f} Lakhs')
print(f'RMSE : Rs. {rmse:.2f} Lakhs')
print(f'R2   : {r2:.4f}')

plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.5, color='#4F46E5', s=30)
lims = [min(y_test.min(), y_pred.min()) - 5, max(y_test.max(), y_pred.max()) + 5]
plt.plot(lims, lims, 'r--', label='Perfect prediction')
plt.xlabel('Actual Price (Rs. Lakhs)')
plt.ylabel('Predicted Price (Rs. Lakhs)')
plt.title('Actual vs Predicted')
plt.legend()
plt.show()

In [ ]:
test_houses = pd.DataFrame([
    {'size_sqft': 1850, 'bedrooms': 3, 'bathrooms': 2,   'age_years': 8,  'location_score': 8.2},
    {'size_sqft': 3500, 'bedrooms': 5, 'bathrooms': 4,   'age_years': 2,  'location_score': 9.5}
])

predictions = model.predict(test_houses)
for i, pred in enumerate(predictions):
    print(f'House {i+1} predicted price: Rs. {pred:.2f} Lakhs')

In [ ]:
os.makedirs('model', exist_ok=True)
joblib.dump(model, 'model/model.pkl')

with open('model/feature_names.json', 'w') as f:
    json.dump(features, f)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
username = api.whoami()['name']
repo_id  = f'{username}/house-price-predictor'

api.create_repo(repo_id=repo_id, exist_ok=True, repo_type='model')
api.upload_folder(
    folder_path='model',
    repo_id=repo_id,
    repo_type='model',
    commit_message='Upload trained Linear Regression model from Colab'
)
print('Uploaded successfully!')